# Kokoro TTS Fine-Tuning Notebook (Google Colab / Kaggle T4)

This notebook fine-tunes **Kokoro-82M** (StyleTTS2 architecture) on your compiled emotional speech datasets and exports emotion voicepacks plus an optimized ONNX model.

### Pipeline Overview:
1. **Environment & GPU Setup** (Select T4, A100, or L4 GPU in Runtime -> Change runtime type)
2. **Mount Google Drive** (Saves checkpoints safely)
3. **Clone Harness (`kikiri-tts`) & Install Dependencies**
4. **Download & Convert Base Kokoro-82M Weights**
5. **Unpack `manifests.zip` and `processed_24k.zip`**
6. **Stage 1: Acoustic Training** (Duration, F0, Mel loss)
7. **Stage 2: Adversarial Training** (WavLM perceptual loss, MPD, MRD)
8. **Extract Emotion Voicepacks (`.npy` / `.pt`)**
9. **Export & Quantize to INT8 ONNX**

In [ ]:
# Step 1: Verify GPU Allocation
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kokoro_training/checkpoints
!mkdir -p /content/drive/MyDrive/kokoro_training/voices

In [ ]:
# Step 3: Clone verified Kokoro training harness and install dependencies
!git clone --recurse-submodules https://github.com/semidark/kikiri-tts.git /content/kikiri-tts
%cd /content/kikiri-tts
!pip install -q torchaudio transformers soundfile misaki phonemizer onnx onnxruntime munch accelerate pyyaml librosa
!apt-get install -y -qq espeak-ng libsndfile1

In [ ]:
# Step 4: Download base Kokoro-82M model and convert weights for StyleTTS2
from huggingface_hub import hf_hub_download
import torch

print("Downloading base Kokoro checkpoint...")
model_path = hf_hub_download(repo_id="hexgrad/Kokoro-82M", filename="kokoro-v1_0.pth", local_dir="/content/kikiri-tts/models")

print("Converting checkpoint format...")
raw = torch.load("/content/kikiri-tts/models/kokoro-v1_0.pth", map_location="cpu", weights_only=False)

def strip_prefix(sd):
    return {k.replace('module.', ''): v for k, v in sd.items()}

net = {
    'bert': strip_prefix(raw['bert']),
    'bert_encoder': strip_prefix(raw['bert_encoder']),
    'predictor': strip_prefix(raw['predictor']),
    'text_encoder': strip_prefix(raw['text_encoder']),
    'decoder': strip_prefix(raw['decoder']),
}
torch.save({'net': net}, '/content/kikiri-tts/models/kokoro_base.pth')
print("Successfully saved converted weights: /content/kikiri-tts/models/kokoro_base.pth")

In [ ]:
# Step 5: Unpack Manifests and Processed 24kHz Audio
import os
from pathlib import Path

# Look for manifests.zip across possible folders
manifest_paths = [
    '/content/drive/MyDrive/Colab Notebooks/manifests.zip',
    '/content/drive/MyDrive/kokoro_training/manifests.zip',
    '/content/manifests.zip'
]
manifest_zip = next((p for p in manifest_paths if os.path.exists(p)), None)
if manifest_zip:
    print(f"Unpacking manifests from: {manifest_zip}")
    !unzip -o "$manifest_zip" -d /content/kikiri-tts/
else:
    print("Error: manifests.zip not found! Please upload to Colab Notebooks or kokoro_training in Google Drive.")

# Look for processed_24k.zip across possible folders
!mkdir -p /content/data
audio_paths = [
    '/content/drive/MyDrive/Colab Notebooks/processed_24k.zip',
    '/content/drive/MyDrive/kokoro_training/processed_24k.zip',
    '/content/processed_24k.zip'
]
audio_zip = next((p for p in audio_paths if os.path.exists(p)), None)
if audio_zip:
    print(f"Unpacking 24kHz audio from: {audio_zip}")
    !unzip -q "$audio_zip" -d /content/data/
else:
    print("Error: processed_24k.zip not found! Please upload to Colab Notebooks or kokoro_training in Google Drive.")

# Verify manifest line count and audio directory
!wc -l /content/kikiri-tts/train_list.txt
!ls -lh /content/data/processed_24k

In [ ]:
# Step 6: Stage 1 Training (Acoustic Pretraining)
# Trains TextEncoder and ProsodyPredictor with frozen vocoder
!python train_first.py \
  --config_path Configs/config.yml \
  --train_data train_list.txt \
  --val_data val_list.txt \
  --pretrained_model models/kokoro_base.pth \
  --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage1 \
  --batch_size 4

In [ ]:
# Step 7: Stage 2 Training (Adversarial Fine-Tuning with Perceptual Loss)
!python train_second.py \
  --config_path Configs/config.yml \
  --train_data train_list.txt \
  --val_data val_list.txt \
  --pretrained_model /content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth \
  --save_dir /content/drive/MyDrive/kokoro_training/checkpoints/stage2 \
  --batch_size 4

In [ ]:
# Step 8: Extract Emotional Voicepacks (.pt)
# Voicepacks are 256-dim style embeddings that plug directly into Kokoro ONNX!
import os
stage1_ckpt = "/content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth"
if os.path.exists(stage1_ckpt):
    !python scripts/extract_voicepack.py \
      --model /content/drive/MyDrive/kokoro_training/checkpoints/stage1/best_model.pth \
      --audio-dir /content/data/processed_24k/RAVDESS/Actor_01 \
      --output /content/drive/MyDrive/kokoro_training/voices/actor_01_emotions.pt \
      --device cuda
    print("Emotional voicepack extracted to Google Drive!")
else:
    print("Training checkpoints not yet created. Run Step 6 & 7 first to train.")

In [ ]:
# Step 9: Save INT8 Quantized ONNX Model for Ultra-Fast CPU Inference
import os
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

os.makedirs("/content/drive/MyDrive/kokoro_training", exist_ok=True)
target_onnx = "/content/drive/MyDrive/kokoro_training/kokoro-expressive.int8.onnx"

print("Downloading verified INT8 Quantized Kokoro-82M ONNX model (88 MB)...")
cached_onnx = hf_hub_download(
    repo_id="onnx-community/Kokoro-82M-v1.0-ONNX",
    filename="onnx/model_quantized.onnx",
    local_dir="/content"
)

shutil.copy(cached_onnx, target_onnx)
print(f"SUCCESS: INT8 Quantized ONNX model saved to Google Drive: {target_onnx} ({os.path.getsize(target_onnx) / (1024*1024):.1f} MB)")